# BrainDecoding — Google Colab GPU 留出测试集 (`test`) 评估与可视化工作流

本 Notebook 专门用于在 **Google Colab GPU 实例** 上对已完成训练的模型执行**留出测试集 (`split=test`) 全面评估与可视化对比**：
- **支持三阶段完整横向对比（`--all-stages`）**：
  - **阶段 1 (`word`)**：词级脑编码器（纯 CNN，`use_transformer: false`）
  - **阶段 2.1 (`context`)**：词级脑编码器 + 16 层全局上下文 Transformer（`use_transformer: true`，完整窗口，无局部滚动修订）
  - **阶段 2.2 (`revision`)**：词级脑编码器 + 16 层局部窗口滚动修订（`use_transformer: true`，`K0/K1/K2` 局部窗口滚动回修）
- **自动输出多维度定量报表与高清可视化图表**：
  1. 多词表规模检索曲线（`N=20, 50, 100, 150, Full Vocab` 的 Macro/Micro Top-10、Top-1、MRR、中位秩 vs 随机基线）；
  2. 分受试者（`sub-01` .. `sub-N`）测试集 50 词检索准确率拆解；
  3. 阶段 2.2 滚动修订各延迟视野（`K0 -> K1 -> K2`）测试集增益拆解；
  4. 阶段 1 vs 阶段 2.1 vs 阶段 2.2 测试集横向对比柱状图与词表规模扩展曲线。

In [ ]:
# 1. 检查 GPU 并挂载 Google Drive、配置环境变量
!nvidia-smi

import os
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('非 Colab 环境，跳过 drive.mount')

# ==================== 用户可修改参数 ====================
GIT_REPO_URL = 'https://github.com/wdongdong0708-jpg/BrainDecoding2.git'
GIT_BRANCH = 'master'
DRIVE_BASE_DIR = '/content/drive/MyDrive/BrainDecoding'

# 选择要评估的数据集或单阶段 YAML 配置文件：
#   - 'configs/training/pallier2025_word_encoder.yaml'          (法语 MEG 阶段 1:   词级脑编码器 CNN)
#   - 'configs/training/pallier2025_context_transformer.yaml'   (法语 MEG 阶段 2.1: 词级脑编码器 + 16层全局上下文 Transformer)
#   - 'configs/training/pallier2025_rolling_revision.yaml'      (法语 MEG 阶段 2.2: 词级脑编码器 + 16层局部窗口滚动修订)
#   - 'configs/training/chineseeeg2_word_encoder.yaml'          (中文 EEG 阶段 1:   词级脑编码器 CNN)
#   - 'configs/training/chineseeeg2_context_transformer.yaml'   (中文 EEG 阶段 2.1: 词级脑编码器 + 16层全局上下文 Transformer)
#   - 'configs/training/chineseeeg2_rolling_revision.yaml'      (中文 EEG 阶段 2.2: 词级脑编码器 + 16层局部窗口滚动修订)
DATASET = 'pallier2025'               # 'pallier2025' 或 'chineseeeg2_littleprince'
CONFIG_FILE = 'configs/training/pallier2025_rolling_revision.yaml'
MODALITY = 'meg'                      # 'meg' (pallier2025) 或 'eeg' (chineseeeg2)
SEED = 0                              # 随机种子: 0, 1, 2
SUBSET_ARGS = '--subject all'         # 必须与训练时的受试者子集保持一致
# ========================================================

CODE_DIR = '/content/code'
VENV_DIR = '/content/proj-venv'
PYTHON = f'{VENV_DIR}/bin/python'
LINKS_DIR = '/content/studies_links'
LOCAL_CACHE_DIR = '/content/local_cache'

DRIVE_DATA_DIR = f'{DRIVE_BASE_DIR}/data'
DRIVE_ARCHIVE_DIR = f'{DRIVE_BASE_DIR}/archives'
EXP_TAG = 'lr1e-4_bs128'  # 自定义超参数实验名称
DRIVE_RESULTS_DIR = f'{DRIVE_BASE_DIR}/runs/{EXP_TAG}'
DRIVE_MODELS_DIR = f'{DRIVE_BASE_DIR}/models'

for folder in (
    LINKS_DIR,
    LOCAL_CACHE_DIR,
    DRIVE_DATA_DIR,
    DRIVE_ARCHIVE_DIR,
    DRIVE_RESULTS_DIR,
    DRIVE_MODELS_DIR,
):
    Path(folder).mkdir(parents=True, exist_ok=True)

os.environ['BRAINDECODING_DATA_DIR'] = LINKS_DIR
os.environ['BRAINDATA_ROOT'] = LINKS_DIR
os.environ['BRAINDECODING_CACHE_DIR'] = LOCAL_CACHE_DIR
os.environ['BRAINDECODING_RESULTS_DIR'] = DRIVE_RESULTS_DIR
os.environ['BRAINDECODING_MODEL_DIR'] = DRIVE_MODELS_DIR

print('DATASET                   =', DATASET)
print('CONFIG_FILE               =', CONFIG_FILE)
print('BRAINDECODING_RESULTS_DIR =', os.environ['BRAINDECODING_RESULTS_DIR'])

In [ ]:
# 2. 克隆或同步最新代码到 /content/code
!if [ ! -d "{CODE_DIR}/.git" ]; then \
    git clone -b {GIT_BRANCH} {GIT_REPO_URL} {CODE_DIR}; \
else \
    git -C {CODE_DIR} fetch origin && git -C {CODE_DIR} checkout {GIT_BRANCH} && git -C {CODE_DIR} pull --ff-only; \
fi

In [ ]:
# 3. 初始化虚拟环境 + 建立 /content/studies_links 软链接桥
!python3 -u {CODE_DIR}/scripts/colab_setup.py \
    --venv-dir {VENV_DIR} \
    --project-root {CODE_DIR} \
    --drive-data-dir {DRIVE_DATA_DIR} \
    --links-dir {LINKS_DIR} \
    --local-cache-dir {LOCAL_CACHE_DIR}

In [ ]:
# 4. 秒级解包 Google Drive 上的 .tar 缓存归档回 /content/local_cache
!cd {CODE_DIR} && {PYTHON} -u scripts/cache_archive.py unpack \
    --archive-dir {DRIVE_ARCHIVE_DIR} \
    --cache-dir {LOCAL_CACHE_DIR} \
    --modality {MODALITY} \
    --skip-existing

In [ ]:
# 5. 【推荐】一键评估当前数据集下所有已训练阶段（阶段 1 word / 阶段 2.1 context / 阶段 2.2 revision）在测试集 (test) 上的表现并绘制对比图
# 注：若早期打包 .tar 时未带 --include-test，不加 --require-cache 会自动补齐缺失的测试集信号缓存
!cd {CODE_DIR} && {PYTHON} -u -m braindecoding eval-run \
    --dataset {DATASET} \
    --seed {SEED} \
    --split test \
    --all-stages \
    {SUBSET_ARGS}

from IPython.display import Image, display

test_pngs = sorted(
    Path(DRIVE_RESULTS_DIR).rglob(f'seed-{int(SEED):03d}/figures/*test*.png'),
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)
shown = set()
for png_path in test_pngs:
    key = (png_path.parent.parent.parent.name, png_path.name)
    if key in shown:
        continue
    shown.add(key)
    print(f'\n[测试集图表] {png_path}')
    display(Image(filename=str(png_path)))

In [ ]:
# 6. 【可选】仅评估单个指定的 CONFIG_FILE 在测试集 (test) 上的表现与可视化看板
!cd {CODE_DIR} && {PYTHON} -u -m braindecoding eval-run \
    --config {CONFIG_FILE} \
    --seed {SEED} \
    --split test \
    {SUBSET_ARGS}

from IPython.display import Image, display

single_pngs = sorted(
    Path(DRIVE_RESULTS_DIR).rglob(f'seed-{int(SEED):03d}/figures/*test*.png'),
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)[:2]
for png_path in single_pngs:
    print(f'\n[单阶段测试集图表] {png_path}')
    display(Image(filename=str(png_path)))